# **UNIVERSIDADE FEDERAL DO CEARA**
---
Disciplina: Introducao a analise em Big Data

---

Professor: Luiz Alexandre

---

Alunos:
1.   Julio Cesar Gama Feitosa Freitas - 583956
2.   Vitoria Freire Rocha Teixeira de Oliveira - 587661

---
Data: 13/09/2026

# 🧪 Lab 9 — Export: tirar a Gold layer do Hive
## 🎯 Objetivo

Levar `gold_fraud_risk` para fora do Hive, testando os 3 caminhos vistos no slide 3 do DIA 3, e escolher qual usar no Lab 11 (Dashboard).

**Rota B - DuckDB + Python/Google Colab**


In [4]:
# Instala o PySpark ANTES de montar a pipeline
# Fazer isso cedo evita que um pedido de "restart runtime" no meio do lab
!pip install pyspark --quiet

In [5]:
# Importa as bibliotecas e cria as pastas utilizadas pelo laboratorio.
import os
import shutil
import duckdb

os.makedirs("bigdata/raw/customers", exist_ok=True)
os.makedirs("bigdata/raw/transactions", exist_ok=True)
os.makedirs("bigdata/bronze", exist_ok=True)
os.makedirs("bigdata/silver", exist_ok=True)
os.makedirs("bigdata/gold", exist_ok=True)

# Abre uma conexao DuckDB.
con = duckdb.connect()

print("Ambiente preparado.")

Ambiente preparado.


## Reconstruir Bronze, Silver e Gold (mesmas regras dos Labs 6 e 7)

In [6]:
# Faz o upload dos CSVs brutos: customers_synthetic.csv e transactions_synthetic.csv
#from google.colab import files

uploaded = [
    "../customers_synthetic.csv",
    "../transactions_synthetic.csv",
    "../fraud_labels.csv"
]

missing = [f for f in uploaded if not os.path.exists(f)] # Verifica se todos os arquivos necessários foram carregados
if missing:
    raise FileNotFoundError("Arquivos ausentes: " + ", ".join(missing))

print("\n✓ Os 3 datasets foram encontrados.")


✓ Os 3 datasets foram encontrados.


In [7]:
# Copia os CSVs enviados para a estrutura Raw do projeto
# Usa "in name" para tolerar sufixos que o Colab adiciona em uploads repetidos,
# como "customers_synthetic (1).csv"
for name in uploaded:
    if "customers_synthetic" in name:
        shutil.copy(name, "bigdata/raw/customers/customers_synthetic.csv")
    elif "transactions_synthetic" in name:
        shutil.copy(name, "bigdata/raw/transactions/transactions_synthetic.csv")

print("Arquivos Raw preparados.")

Arquivos Raw preparados.


In [8]:
# Recria a Bronze de clientes e de transacoes com as mesmas regras do Lab 6
customers_raw = "bigdata/raw/customers/customers_synthetic.csv"
transactions_raw = "bigdata/raw/transactions/transactions_synthetic.csv"

con.sql(f"""
CREATE OR REPLACE TABLE bronze_customers AS
SELECT DISTINCT
    customer_id, name, cpf, email, segment,
    CAST(credit_score AS INT) AS credit_score,
    CAST(created_at AS DATE) AS created_at
FROM read_csv_auto('{customers_raw}')
WHERE customer_id IS NOT NULL
  AND credit_score BETWEEN 300 AND 900
""")

# O CASE converte explicitamente True/False (texto) para BOOLEAN
con.sql(f"""
CREATE OR REPLACE TABLE bronze_transactions AS
SELECT DISTINCT
    transaction_id, customer_id,
    CAST(amount AS FLOAT) AS amount,
    transaction_type, status,
    CAST(risk_score AS FLOAT) AS risk_score,
    CASE WHEN is_fraud = 'True' THEN true ELSE false END AS is_fraud,
    CAST(timestamp AS TIMESTAMP) AS ts
FROM read_csv_auto('{transactions_raw}')
WHERE amount > 0
  AND customer_id IS NOT NULL
""")

con.sql("SELECT COUNT(*) AS total FROM bronze_customers").show()
con.sql("SELECT COUNT(*) AS total FROM bronze_transactions").show()

┌───────┐
│ total │
│ int64 │
├───────┤
│  9993 │
└───────┘

┌────────┐
│ total  │
│ int64  │
├────────┤
│ 100000 │
└────────┘



In [9]:
# Recria a Silver juntando transacoes com clientes e derivando as colunas de analise
con.sql("""
CREATE OR REPLACE TABLE silver_transactions AS
SELECT
  t.transaction_id, t.customer_id, t.amount, t.transaction_type,
  t.status, t.risk_score, t.is_fraud, t.ts,
  c.segment, c.credit_score,
  year(t.ts)  AS year,
  month(t.ts) AS month,
  day(t.ts)   AS day,
  dayofweek(t.ts) AS day_of_week,
  CASE
    WHEN t.amount < 100  THEN 'baixo'
    WHEN t.amount < 1000 THEN 'medio'
    ELSE 'alto'
  END AS amount_band
FROM bronze_transactions t
JOIN bronze_customers c ON t.customer_id = c.customer_id
""")

con.sql("SELECT COUNT(*) FROM silver_transactions").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       100000 │
└──────────────┘



In [10]:
# Recria a Gold gold_fraud_risk com as mesmas regras do Lab 7
# E a unica Gold necessaria para este lab - gold_daily_metrics nao e usada no export
con.sql("""
CREATE OR REPLACE TABLE gold_fraud_risk AS
SELECT
  segment,
  COUNT(*) AS total_transacoes,
  SUM(amount) AS valor_total,
  ROUND(AVG(amount), 2) AS ticket_medio,
  SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS qtd_fraudes,
  ROUND(100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*), 2) AS taxa_fraude_pct,
  SUM(CASE WHEN is_fraud THEN amount ELSE 0 END) AS valor_em_risco
FROM silver_transactions
GROUP BY segment
""")

con.sql("SELECT * FROM gold_fraud_risk ORDER BY taxa_fraude_pct DESC").show()

┌───────────┬──────────────────┬────────────────────┬──────────────┬─────────────┬─────────────────┬────────────────────┐
│  segment  │ total_transacoes │    valor_total     │ ticket_medio │ qtd_fraudes │ taxa_fraude_pct │   valor_em_risco   │
│  varchar  │      int64       │       double       │    double    │   int128    │     double      │       double       │
├───────────┼──────────────────┼────────────────────┼──────────────┼─────────────┼─────────────────┼────────────────────┤
│ High-Risk │             9155 │ 1664640.2447309494 │       181.83 │         705 │             7.7 │ 126451.06910800934 │
│ Standard  │            29689 │  5479399.846734047 │       184.56 │         655 │            2.21 │ 106828.41910934448 │
│ Premium   │            61156 │ 11235041.576210976 │       183.71 │         473 │            0.77 │  81308.43799591064 │
└───────────┴──────────────────┴────────────────────┴──────────────┴─────────────┴─────────────────┴────────────────────┘



## Opção 1 — CSV direto do DuckDB (equivalente à Opção 2 da rota A)

In [11]:
# Exporta a Gold direto para CSV
con.sql("COPY gold_fraud_risk TO 'fraud_risk_export.csv' (HEADER, DELIMITER ',')")

# Confere o conteudo do arquivo gerado
with open("fraud_risk_export.csv") as f:
    print(f.read())

segment,total_transacoes,valor_total,ticket_medio,qtd_fraudes,taxa_fraude_pct,valor_em_risco
Standard,29689,5479399.846734047,184.56,655,2.21,106828.41910934448
Premium,61156,11235041.576210976,183.71,473,0.77,81308.43799591064
High-Risk,9155,1664640.2447309494,181.83,705,7.7,126451.06910800934



## Opção 2 — SQLite (equivalente ao Sqoop export para MySQL)

In [12]:
# Exporta a Gold para uma tabela em um banco SQLite local
import sqlite3
import pandas as pd

df = con.sql("SELECT * FROM gold_fraud_risk").df()

conn = sqlite3.connect("bi_db.sqlite")
df.to_sql("fraud_risk_bi", conn, if_exists="replace", index=False)

print("Exportado para bi_db.sqlite - tabela fraud_risk_bi")

Exportado para bi_db.sqlite - tabela fraud_risk_bi


In [13]:
# Confere que a tabela foi gravada corretamente no SQLite
check = pd.read_sql("SELECT * FROM fraud_risk_bi", conn)
print(check)

     segment  total_transacoes   valor_total  ticket_medio  qtd_fraudes  \
0   Standard             29689  5.479400e+06        184.56        655.0   
1    Premium             61156  1.123504e+07        183.71        473.0   
2  High-Risk              9155  1.664640e+06        181.83        705.0   

   taxa_fraude_pct  valor_em_risco  
0             2.21   106828.419109  
1             0.77    81308.437996  
2             7.70   126451.069108  


## Opção 3 — PySpark local lendo o Parquet (equivalente à Opção 3 da rota A)

In [14]:
# Checagem de seguranca: garante que o Parquet da Gold existe em disco
# antes de o Spark tentar le-lo
if not os.path.exists("bigdata/gold/fraud_risk.parquet"):
    print("Parquet nao encontrado - recriando a partir de gold_fraud_risk...")
    con.sql("COPY gold_fraud_risk TO 'bigdata/gold/fraud_risk.parquet' (FORMAT PARQUET)")

print("OK:", os.path.exists("bigdata/gold/fraud_risk.parquet"))

Parquet nao encontrado - recriando a partir de gold_fraud_risk...
OK: True


In [15]:
# Le o Parquet da Gold com Spark, fora do DuckDB, para comparar o resultado
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("export").master("local[*]").getOrCreate()

df_spark = spark.read.parquet("bigdata/gold/fraud_risk.parquet")
df_spark.show()

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
# Grava a mesma tabela no SQLite a partir do Spark, para fechar o paralelo com a Rota A
df_spark.toPandas().to_sql("fraud_risk_bi_spark", conn, if_exists="replace", index=False)

print("Exportado para bi_db.sqlite - tabela fraud_risk_bi_spark")
conn.close()

Exportado para bi_db.sqlite - tabela fraud_risk_bi_spark


## Checkpoint

- [ ] `gold_fraud_risk` reconstruida nesta sessao a partir dos CSVs brutos
- [ ] Testou pelo menos 2 dos 3 caminhos de export
- [ ] `fraud_risk_export.csv` existe e tem 3 linhas de dados
- [ ] Decidiu qual caminho usar no Lab 11 (recomendacao: CSV, e o mais simples de importar no Metabase)
- [ ] Parquet da Gold e arquivos de export baixados para a maquina local

---

**Proximo lab:** `DIA3_LAB10_SPARK.md` - explorar com Spark SQL.